# 11.10 - Reranking

**Phase:** 11 - RAG Systems

**Status:** VERIFIED

---

## 1. What Are We Solving?

First-stage retrieval (vector/keyword) is fast but approximate. A cross-encoder that reads the query and each candidate *together* produces far more accurate relevance scores - so we retrieve a broad set, then rerank to a tight top-k.

## 2. Why Does This Matter?

Reranking is the biggest single precision win in a RAG pipeline. It lifts the top results above noise while adding only modest latency.

## 3. Prerequisites

Unit 11.9 (Retrieval).

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Build a two-stage pipeline: retrieve top-20 from Chroma, rerank to top-3
- Use a CrossEncoder when available, else a lexical+vector blend fallback
- Compare before/after on a couple of hard queries

## 5. Mental Model

Reranking is a senior librarian reviewing initial search results: the first search finds candidates, the senior reads query+doc together and picks the best.

```text
Query -> Retrieve top-20 (fast, approximate) -> Rerank -> Return top-3 (precise)
```


## 6. Setup + Index
Small corpus of claims where keyword and vector scores are ambiguous - good bait for a reranker to sort out.

In [1]:
# Deterministic embedding helper.
# Loads all-MiniLM-L6-v2 if available; otherwise falls back to a hash-based
# vector so every cell still completes offline. The fallback still gives
# "similar text -> similar vector" behaviour via character-bigram overlap,
# so the demos remain meaningful without the model download.
import hashlib, numpy as np

_DIM = 384


def _hash_embed(texts):
    vecs = np.zeros((len(texts), _DIM))
    for i, t in enumerate(texts):
        bigrams = [t[j:j+2].lower() for j in range(len(t)-1)]
        for bg in bigrams:
            h = int(hashlib.md5(bg.encode()).hexdigest(), 16) % _DIM
            vecs[i, h] += 1.0
        norm = np.linalg.norm(vecs[i]) or 1.0
        vecs[i] = vecs[i] / norm
    return vecs


_model = None
_model_name = "all-MiniLM-L6-v2"


def get_embedder(force_fallback=False):
    """Return a function texts -> np.ndarray (N, dim)."""
    global _model
    if force_fallback:
        return _hash_embed
    if _model is None:
        try:
            from sentence_transformers import SentenceTransformer
            _model = SentenceTransformer(_model_name)
        except Exception as e:
            print("MiniLM unavailable, using hash fallback:", type(e).__name__)
            _model = None
    if _model is None:
        return _hash_embed
    return lambda texts: np.asarray(_model.encode(list(texts), convert_to_numpy=True))


def embed(texts, force_fallback=False):
    fn = get_embedder(force_fallback=force_fallback)
    return np.asarray(fn(texts), dtype=np.float32)


print("embedding dim:", _DIM)
print("backend:", _model_name if get_embedder() != _hash_embed else "hash-fallback")


embedding dim: 384


D:\CODE\complete ml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1868.95it/s]

backend: all-MiniLM-L6-v2


In [2]:
import chromadb

CORPUS = [
    "We offer free standard shipping on all orders over fifty dollars.",
    "Standard shipping takes five to seven business days.",
    "Express shipping costs a flat fee and arrives in two days.",
    "Free shipping is only available within the continental US.",
    "International orders must pay customs duties at delivery.",
    "Refunds are issued after the returned item passes inspection.",
    "Returned items must be in the original packaging.",
    "Replacements ship free of charge for defective products.",
    "Warranty covers manufacturing defects for one year.",
    "You have thirty days from purchase to request a return.",
]
ids = [f"c{i}" for i in range(len(CORPUS))]
client = chromadb.Client()
col = client.create_collection("rerank_idx", embedding_function=None,
                               metadata={"hnsw:space": "cosine"})
emb = embed(CORPUS)
col.add(documents=CORPUS, embeddings=emb.tolist(), ids=ids)
print("indexed", col.count())


indexed 10


## 7. Two-Stage: Retrieve 20 -> Rerank to 3
Stage 1 pulls a broad top-20 by vector similarity. Stage 2 optionally uses a `CrossEncoder` to score each (query, doc) pair and returns the top-3. If the cross-encoder isn't installed/downloaded, we fall back to a lexical+vector blend (keyword overlap + cosine), so the notebook still runs.

In [3]:
import numpy as np


def first_stage(query, k=20):
    r = col.query(query_embeddings=embed([query]).tolist(), n_results=k)
    return [(cid, doc) for cid, doc in zip(r["ids"][0], r["documents"][0])]


def make_reranker():
    try:
        from sentence_transformers import CrossEncoder
        ce = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
        return lambda q, cands: ce.predict([(q, d) for _, d in cands])
    except Exception as e:
        print("cross-encoder unavailable -> lexical+vector blend fallback:", type(e).__name__)

        def blend(q, cands):
            q_tok = set(q.lower().replace(".", "").split())
            scores = []
            for cid, doc in cands:
                d_tok = set(doc.lower().replace(".", "").split())
                overlap = len(q_tok & d_tok) / max(1, len(q_tok))
                # cosine from the index via embedding
                cos = float(np.dot(embed([q])[0], embed([doc])[0]))
                scores.append(0.5 * overlap + 0.5 * cos)
            return np.array(scores)
        return blend


rerank = make_reranker()


def two_stage(query, first_k=20, final_k=3):
    cands = first_stage(query, first_k)
    scores = rerank(query, cands)
    order = np.argsort(-np.asarray(scores))
    return [cands[i] for i in order[:final_k]]


D:\CODE\complete ml\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PC\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1447.80it/s]

## 8. Compare Before / After on a Hard Query
Query: 'free shipping overseas'. Vector alone may surface the right idea; reranking pulls the truly relevant 'customs duties' / 'only continental US' claims to the top.

In [4]:
def show_stage(query):
    cands = first_stage(query, 20)
    s1 = [c[1] for c in cands[:3]]
    s2 = [c[1] for c in two_stage(query)]
    print("QUERY:", query)
    print("  STAGE-1 (vector top-3):")
    for d in s1:
        print("    -", d)
    print("  STAGE-2 (after rerank top-3):")
    for d in s2:
        print("    -", d)
    print()


for q in ["free shipping overseas", "return broken item after a year"]:
    show_stage(q)


QUERY: free shipping overseas
  STAGE-1 (vector top-3):
    - Free shipping is only available within the continental US.
    - We offer free standard shipping on all orders over fifty dollars.
    - International orders must pay customs duties at delivery.
  STAGE-2 (after rerank top-3):
    - Free shipping is only available within the continental US.
    - We offer free standard shipping on all orders over fifty dollars.
    - Replacements ship free of charge for defective products.



QUERY: return broken item after a year
  STAGE-1 (vector top-3):
    - You have thirty days from purchase to request a return.
    - Warranty covers manufacturing defects for one year.
    - Returned items must be in the original packaging.
  STAGE-2 (after rerank top-3):
    - Returned items must be in the original packaging.
    - Refunds are issued after the returned item passes inspection.
    - You have thirty days from purchase to request a return.



## 9. Latency vs Quality Trade-Off
Reranking is slower per candidate than vector similarity, so you rerank only the broad shortlist (20-50), not thousands. The fixture below shows the shape of the trade-off schematically.

In [5]:
import pandas as pd
trade = pd.DataFrame([
    ["Stage-1 vector", "ms", "approximate", "10ms for 50"],
    ["Stage-2 rerank", "ms per pair", "precise", "20-50 candidates"],
    ["Full pipeline", "~200ms", "best quality", "retrieve 50 -> rerank -> top-5"],
], columns=["Step", "Latency", "Precision", "Cost notes"])
print(trade.to_string(index=False))


          Step     Latency    Precision                     Cost notes
Stage-1 vector          ms  approximate                    10ms for 50
Stage-2 rerank ms per pair      precise               20-50 candidates
 Full pipeline      ~200ms best quality retrieve 50 -> rerank -> top-5



## Common Mistakes

- Reranking too many candidates (slow, diminishing returns).
- Not reranking at all (leaving precision on the table).
- Using the same model for retrieval and reranking (wastes compute).
- Ignoring latency impact on UX.

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| Reranking slow | Too many candidates | Reduce to 20-50 |
| No quality improvement | Cross-encoder unsuited to domain | Try a different reranker |
| Memory errors | Too many candidates / big model | Reduce batch or candidates |

## Best Practices

- Rerank 20-50 candidates, not thousands.
- Use reranking as a second stage, not primary retrieval.
- Measure quality gain vs latency cost.
- Fall back gracefully when the reranker model is absent.

## Hands-On Practice

1. **Basic:** Rerank 5 candidates with a cross-encoder.
2. **Guided:** Compare top-5 from vector vs from reranking.
3. **Independent:** Build retrieve-50 -> rerank-to-5.
4. **Realistic:** Measure latency and quality improvement.
5. **Challenge:** Test 3 rerankers on the same query set.

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.
